In [ ]:
# CELL 1 — Verify all libraries are available
import sys
print(f"Python version: {sys.version}")

libraries = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "joblib"]
for lib in libraries:
    try:
        __import__(lib)
        print(f"  ✅ {lib} is installed")
    except ImportError:
        print(f"  ❌ {lib} is MISSING — run: pip install {lib}")


In [ ]:
# CELL 2 — Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib

print("✅ All libraries imported successfully!")


In [ ]:
# CELL 3 — Load the LIGTAS voltage dataset
df = pd.read_csv("ligtas_voltage_dataset.csv")

print("=== DATASET LOADED ===")
print(f"Shape         : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print("\nFirst 10 rows:")
display(df.head(10))
print("\nLabel distribution:")
print(df['status'].value_counts())


In [ ]:
# CELL 4 — Gap fixes applied (verification)
#
# ┌─────────────────────────────────────────────────────────────────┐
# │  GAP 1 FIX — No distance assumption in formula                  │
# │                                                                 │
# │  OLD (WRONG): radius = 100 / (1 + voltage)                      │
# │               → assumed high voltage = near source              │
# │               → assumed low voltage  = far source               │
# │               → BOTH assumptions are WRONG                      │
# │                                                                 │
# │  NEW (FIXED): coverage_area = π × voltage²                      │
# │               → higher voltage = more energy = larger zone      │
# │               → we make NO claim about distance                 │
# │               → a weak source CAN be nearby                     │
# │               → a strong source CAN be far away                 │
# │               → the sensor only tells us intensity, not distance │
# ├─────────────────────────────────────────────────────────────────┤
# │  GAP 3 FIX — Coverage area is physically grounded               │
# │                                                                 │
# │  A = π × V² is derived from electric field theory:              │
# │  AC leakage in water radiates energy outward.                   │
# │  The affected area scales with V² (energy proportional to V²).  │
# │  voltage_squared added as explicit feature so model sees         │
# │  both V and V² separately.                                      │
# │  This is a theoretical ESTIMATE, clearly stated as such.        │
# ├─────────────────────────────────────────────────────────────────┤
# │  GAP 4 FIX — Temporal features added                            │
# │                                                                 │
# │  voltage_delta  : change from previous reading                  │
# │                   rising = possible fault developing            │
# │                   falling = leakage dissipating                 │
# │  voltage_trend  : 1=rising, 0=stable, -1=falling                │
# │  spike_flag     : 1 if sudden jump >10V (surge/fault event)     │
# │                   A 20V spike to 35V is more dangerous          │
# │                   than a stable 35V reading                     │
# ├─────────────────────────────────────────────────────────────────┤
# │  GAP 6 FIX — Realistic sensor noise simulation                  │
# │                                                                 │
# │  sensor_noise_level : ZMPT101B noise (~2% of voltage reading)   │
# │  Borderline 25–35V  : oversampled ×30 with ±1.5V noise          │
# │                        → model learns uncertainty at threshold   │
# │  Safe zone 0–5V     : oversampled with fine 0.05V steps         │
# │  Check zone 5–30V   : oversampled with noise at each step       │
# │  Spike outliers     : 80 random interference events added       │
# └─────────────────────────────────────────────────────────────────┘

print("=== GAP FIX VERIFICATION ===")
print()

# GAP 1 check
r30 = df[df['voltage_v'] == 30.0].iloc[0]
import numpy as np
expected = round(np.pi * 30**2, 2)
print(f"GAP 1 — Formula A = π × V²:")
print(f"  V=30  coverage_area = {r30['coverage_area_m2']}  expected = {expected}")
print(f"  ✅ CORRECT — no distance assumption" if abs(r30['coverage_area_m2'] - expected) < 1 else "  ❌ CHECK FORMULA")

# GAP 3 check
print(f"\nGAP 3 — voltage_squared column:")
print(f"  V=10  voltage_squared = {df[df['voltage_v']==10.0].iloc[0]['voltage_squared']}  expected = 100.0")
print(f"  ✅ CORRECT" if df[df['voltage_v']==10.0].iloc[0]['voltage_squared'] == 100.0 else "  ❌ CHECK")

# GAP 4 check
print(f"\nGAP 4 — Temporal features:")
print(f"  voltage_delta  present: {'voltage_delta' in df.columns}")
print(f"  spike_flag     present: {'spike_flag' in df.columns}")
print(f"  voltage_trend  present: {'voltage_trend' in df.columns}")
print(f"  Total spike events    : {df['spike_flag'].sum()}")

# GAP 6 check
border = df[(df['voltage_v'] >= 25) & (df['voltage_v'] <= 35)]
safe   = df[df['status'] == 'Safe']
check  = df[df['status'] == 'Check']
print(f"\nGAP 6 — Noise & oversampling:")
print(f"  sensor_noise_level    : {df['sensor_noise_level'].describe().to_dict()}")
print(f"  Borderline 25–35V rows: {len(border)}")
print(f"  Safe rows             : {len(safe)}")
print(f"  Check rows            : {len(check)}")
print(f"  Dangerous rows        : {len(df[df['status']=='Dangerous'])}")


In [ ]:
# CELL 5 — Data exploration
print("=== DATASET SUMMARY ===")
display(df.describe())

print("\n=== CLASS DISTRIBUTION ===")
print(df['status'].value_counts())
print("\n=== RISK FLAG DISTRIBUTION ===")
print(df['risk_flag'].value_counts())
print(f"\n=== TEMPORAL SUMMARY ===")
print(f"  Spikes (>10V jump)   : {df['spike_flag'].sum()}")
print(f"  Rising readings      : {(df['voltage_trend']==1).sum()}")
print(f"  Falling readings     : {(df['voltage_trend']==-1).sum()}")
print(f"  Stable readings      : {(df['voltage_trend']==0).sum()}")


In [ ]:
# CELL 6 — Data visualization (all gap fixes shown)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("LIGTAS Dataset — All Gap Fixes Applied", fontsize=14, fontweight="bold")

# Plot 1: Voltage distribution
axes[0,0].hist(df["voltage_v"], bins=60, color="steelblue", edgecolor="black", alpha=0.8)
axes[0,0].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
axes[0,0].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[0,0].set_title("Voltage Distribution")
axes[0,0].set_xlabel("Voltage (V)")
axes[0,0].set_ylabel("Count")
axes[0,0].legend()

# Plot 2: Class distribution
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
counts = df["status"].value_counts().reindex(["Safe","Check","Dangerous"])
axes[0,1].bar(counts.index, counts.values, color=colors, edgecolor="black")
axes[0,1].set_title("Class Distribution (3 Classes, Balanced)")
axes[0,1].set_xlabel("Status")
axes[0,1].set_ylabel("Number of Samples")
for i, v in enumerate(counts.values):
    axes[0,1].text(i, v + 30, str(v), ha="center", fontweight="bold")

# Plot 3: GAP 1 & 3 — A = π × V² curve
base = df.drop_duplicates("voltage_v").sort_values("voltage_v")
sample = base[base["voltage_v"] <= 100]
axes[0,2].plot(sample["voltage_v"], sample["coverage_area_m2"], color="steelblue", linewidth=2)
axes[0,2].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
axes[0,2].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=(sample["voltage_v"] < 5),   color="green",  alpha=0.2, label="Safe")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=((sample["voltage_v"] >= 5) & (sample["voltage_v"] < 30)),
                        color="orange", alpha=0.2, label="Check")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=(sample["voltage_v"] >= 30), color="red",    alpha=0.2, label="Dangerous")
axes[0,2].set_title("A = π × V²  (GAP 1 & 3 Fixed — No Distance Assumption)")
axes[0,2].set_xlabel("Voltage V")
axes[0,2].set_ylabel("Coverage Area (m²)")
axes[0,2].legend()
axes[0,2].grid(True, linestyle="--", alpha=0.5)

# Plot 4: GAP 4 — voltage delta distribution
axes[1,0].hist(df["voltage_delta"], bins=60, color="#9b59b6", edgecolor="black", alpha=0.8)
axes[1,0].axvline(x= 10, color="red", linestyle="--", linewidth=2, label="Spike +10V")
axes[1,0].axvline(x=-10, color="red", linestyle="--", linewidth=2, label="Spike -10V")
axes[1,0].set_title("Voltage Delta (GAP 4 Fixed — Temporal Feature)")
axes[1,0].set_xlabel("ΔV from previous reading")
axes[1,0].set_ylabel("Count")
axes[1,0].legend()

# Plot 5: GAP 6 — sensor noise level
axes[1,1].hist(df["sensor_noise_level"], bins=50, color="#e67e22", edgecolor="black", alpha=0.8)
axes[1,1].set_title("Sensor Noise Level (GAP 6 Fixed — Realistic Noise)")
axes[1,1].set_xlabel("Noise Level (V)")
axes[1,1].set_ylabel("Count")

# Plot 6: GAP 6 — borderline zone oversampling
border = df[(df["voltage_v"] >= 25) & (df["voltage_v"] <= 35)]
colors_map = {1: "orange", 2: "red"}
axes[1,2].scatter(border["voltage_v"], border["coverage_area_m2"],
                   c=border["label"].map(colors_map), alpha=0.35, s=12)
axes[1,2].axvline(x=30, color="black", linestyle="--", linewidth=2, label="30V threshold")
axes[1,2].set_title(f"Borderline Zone 25–35V (GAP 6 Fixed — {len(border)} rows)")
axes[1,2].set_xlabel("Voltage V")
axes[1,2].set_ylabel("Coverage Area (m²)")
axes[1,2].legend()
axes[1,2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("ligtas_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart saved as ligtas_overview.png")


In [ ]:
# CELL 7 — Prepare features (X) and labels (y)
#
# 8 features — one for each gap fix:
#
#  Feature              Gap Fixed   Reason
#  ───────────────────  ─────────   ──────────────────────────────────────
#  voltage_v            GAP 1       Raw sensor reading — no distance claim
#  voltage_squared      GAP 1 & 3   V² — energy scales with square of V
#  coverage_area_m2     GAP 1 & 3   π × V² — electrified area, no distance
#  voltage_class        (original)  0–4 intensity category
#  danger_score         (original)  V / (1 + log(area+1)) — combined risk
#  voltage_delta        GAP 4       Change from previous reading
#  spike_flag           GAP 4       Sudden jump >10V detected
#  sensor_noise_level   GAP 6       Estimated ZMPT101B noise level

X = df[[
    "voltage_v",
    "voltage_squared",
    "coverage_area_m2",
    "voltage_class",
    "danger_score",
    "voltage_delta",
    "spike_flag",
    "sensor_noise_level"
]]
y = df["label"]   # 0=Safe  1=Check  2=Dangerous

print("=== FEATURES (X) — 8 features ===")
display(X.head(10))
print("\n=== LABELS (y) ===")
print(f"  0 = Safe      : {(y==0).sum():,}")
print(f"  1 = Check     : {(y==1).sum():,}")
print(f"  2 = Dangerous : {(y==2).sum():,}")
print(f"\n  Total         : {len(y):,}")


In [ ]:
# CELL 8 — Train/Test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("=== TRAIN / TEST SPLIT ===")
print(f"  Training: {len(X_train):,}  (80%)")
print(f"  Testing : {len(X_test):,}  (20%)")
print("\nTrain distribution:")
for lbl, name in [(0,"Safe"),(1,"Check"),(2,"Dangerous")]:
    print(f"  {name:<10}: {(y_train==lbl).sum():,}")
print("\nTest distribution:")
for lbl, name in [(0,"Safe"),(1,"Check"),(2,"Dangerous")]:
    print(f"  {name:<10}: {(y_test==lbl).sum():,}")


In [ ]:
# CELL 9 — Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Features scaled (8 features)")
print("\nScaler means:")
for col, mean in zip(X.columns, scaler.mean_):
    print(f"  {col:<22}: {mean:.4f}")


In [ ]:
# CELL 10 — Train Random Forest model
print("Training Random Forest (100 trees, 8 features)...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"   # handles Safe/Check/Dangerous imbalance
)
rf_model.fit(X_train_scaled, y_train)

print("✅ Random Forest training complete!")
print(f"   Trees    : {rf_model.n_estimators}")
print(f"   Features : {rf_model.n_features_in_}")
print(f"   Classes  : {list(rf_model.classes_)}")

# Feature importance — shows which gap fix contributes most
importances = rf_model.feature_importances_
print("\n=== FEATURE IMPORTANCE (higher = more influential) ===")
for name, imp in sorted(zip(X.columns, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 60)
    print(f"  {name:<22}: {imp:.4f}  {bar}")


In [ ]:
# CELL 11 — Evaluate Random Forest model
rf_pred = rf_model.predict(X_test_scaled)

rf_acc  = accuracy_score(y_test,  rf_pred)
rf_prec = precision_score(y_test, rf_pred, average="weighted", zero_division=0)
rf_rec  = recall_score(y_test,    rf_pred, average="weighted", zero_division=0)
rf_f1   = f1_score(y_test,        rf_pred, average="weighted", zero_division=0)

print("=" * 55)
print("  MODEL: Random Forest — 8 features, all gaps fixed")
print("=" * 55)
print(f"  Accuracy   : {rf_acc:.4f}  ({rf_acc*100:.2f}%)")
print(f"  Precision  : {rf_prec:.4f}")
print(f"  Recall     : {rf_rec:.4f}")
print(f"  F1-Score   : {rf_f1:.4f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, rf_pred,
      target_names=["Safe (0)","Check (1)","Dangerous (2)"]))


In [ ]:
# CELL 12 — Model Metrics Chart
metrics   = ["Accuracy", "Precision", "Recall", "F1-Score"]
rf_scores = [rf_acc, rf_prec, rf_rec, rf_f1]
bar_colors = ["#3b82f6", "#22c55e", "#f59e0b", "#ef4444"]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

bars = ax.bar(metrics, rf_scores, color=bar_colors, edgecolor="white",
              linewidth=1.2, width=0.5, zorder=3)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.4f}", ha="center", va="bottom",
            fontsize=11, color="white", fontweight="bold")

ax.set_ylim(0, 1.15)
ax.set_xticklabels(metrics, color="#e5e7eb", fontsize=12, fontweight="bold")
ax.set_ylabel("Score", color="#9ca3af", fontsize=11)
ax.set_title("LIGTAS — Random Forest Performance (All Gaps Fixed)",
             color="#f9fafb", fontsize=13, fontweight="bold", pad=14)
ax.tick_params(colors="#6b7280")
for spine in ax.spines.values(): spine.set_color("#30363d")
ax.set_yticks([0.0,0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.0","0.2","0.4","0.6","0.8","1.0"], color="#6b7280")
ax.axhline(y=1.0, color="#374151", linestyle=":", linewidth=1.2, zorder=2)
ax.grid(axis="y", color="#21262d", linewidth=0.8, linestyle="--", zorder=1)

plt.tight_layout()
plt.savefig("model_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Metrics chart saved as model_metrics.png")


In [ ]:
# CELL 13 — Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle("LIGTAS — Confusion Matrix (Random Forest, Gap-Fixed)",
             fontsize=12, fontweight="bold")

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Safe","Check","Dangerous"],
            yticklabels=["Safe","Check","Dangerous"],
            linewidths=0.5)
ax.set_xlabel("Predicted Label", fontsize=10)
ax.set_ylabel("Actual Label",    fontsize=10)

print("Per-class summary:")
for i, cls in enumerate(["Safe","Check","Dangerous"]):
    tp = cm[i,i]
    fp = cm[:,i].sum() - tp
    fn = cm[i,:].sum() - tp
    tn = cm.sum() - tp - fp - fn
    print(f"  {cls:<10}: TP={tp}  TN={tn}  FP={fp}  FN={fn}")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved as confusion_matrix.png")


In [ ]:
# CELL 14 — Coverage Area Curve (A = π × V², GAP 1 & 3 Fixed)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("LIGTAS — Coverage Area  A = π × V²  (No Distance Assumption)",
             fontsize=13, fontweight="bold")

base = df.drop_duplicates("voltage_v").sort_values("voltage_v")

for ax, (title, data) in zip(axes, [
    ("Full Range (0–1000V)", base),
    ("Zoomed (0–100V)",      base[base["voltage_v"] <= 100])
]):
    ax.plot(data["voltage_v"], data["coverage_area_m2"], color="steelblue", linewidth=2)
    ax.axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
    ax.axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=(data["voltage_v"] < 5),   color="green",  alpha=0.25, label="Safe")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=((data["voltage_v"] >= 5) & (data["voltage_v"] < 30)),
                     color="orange", alpha=0.2,  label="Check")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=(data["voltage_v"] >= 30), color="red",    alpha=0.2,  label="Dangerous")
    ax.set_title(title)
    ax.set_xlabel("Voltage V (intensity — not distance)")
    ax.set_ylabel("Coverage Area A (m²)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("coverage_area_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Coverage curve saved as coverage_area_curve.png")


In [ ]:
# CELL 15 — Save trained model and scaler
joblib.dump(rf_model, "ligtas_rf_model.pkl")
joblib.dump(scaler,   "ligtas_scaler.pkl")
print("✅ Saved:")
print("   ligtas_rf_model.pkl — Random Forest (8 features)")
print("   ligtas_scaler.pkl   — StandardScaler (8 features)")


In [ ]:
# CELL 16 — Real-time prediction function
# Accepts voltage + previous_voltage for temporal feature (GAP 4)
# No distance assumption — only voltage intensity used (GAP 1)

def ligtas_predict(voltage_value, prev_voltage=None):
    prev_v        = prev_voltage if prev_voltage is not None else voltage_value
    voltage_delta = round(voltage_value - prev_v, 4)
    spike_flag    = 1 if abs(voltage_delta) > 10.0 else 0

    v_sq        = voltage_value ** 2
    area        = np.pi * v_sq
    vc          = (0 if voltage_value < 5 else
                   1 if voltage_value < 15 else
                   2 if voltage_value < 30 else
                   3 if voltage_value < 100 else 4)
    danger_score = voltage_value / (1.0 + np.log1p(area + 1))
    noise_level  = voltage_value * 0.02

    input_scaled = scaler.transform([[
        voltage_value, v_sq, area, vc, danger_score,
        voltage_delta, spike_flag, noise_level
    ]])
    pred = rf_model.predict(input_scaled)[0]

    icons = {0: "🟢 SAFE", 1: "🟡 CHECK", 2: "🔴 DANGEROUS"}
    print(f"  ┌──────────────────────────────────────────────────")
    print(f"  │  Voltage Now      : {voltage_value} V")
    print(f"  │  Previous Reading : {prev_v} V")
    print(f"  │  Coverage Area    : {area:,.2f} m²  (A=π×{voltage_value}²)")
    print(f"  │  Voltage Delta    : {voltage_delta:+.2f} V  {'⚠️ SPIKE' if spike_flag else '(normal)'}")
    print(f"  │  Classification   : {icons[pred]}")
    print(f"  └──────────────────────────────────────────────────")

print("=" * 55)
print("  LIGTAS — Prediction Test (8 features)")
print("=" * 55)
# Test cases including spikes to demonstrate GAP 4 fix
test_cases = [
    (0.0,   None,   "baseline no leakage"),
    (5.0,   0.0,    "weak leakage just started"),
    (10.0,  5.0,    "gradually rising"),
    (25.0,  10.0,   "climbing toward threshold"),
    (29.9,  25.0,   "just below threshold"),
    (30.0,  29.9,   "just crossed threshold"),
    (35.0,  10.0,   "spike from 10V — surge event"),
    (50.0,  49.0,   "steady high voltage"),
    (220.0, 219.0,  "mains voltage leakage"),
]
for v, prev, desc in test_cases:
    print(f"\n  [{desc}]")
    ligtas_predict(v, prev)


In [ ]:
# CELL 17 — Load saved model without retraining
loaded_model  = joblib.load("ligtas_rf_model.pkl")
loaded_scaler = joblib.load("ligtas_scaler.pkl")
print("✅ Model and scaler loaded!")

# Quick test — spike scenario
test_v, prev_v = 35.0, 10.0
v_sq   = test_v ** 2
area   = np.pi * v_sq
vc     = 3
ds     = test_v / (1.0 + np.log1p(area + 1))
delta  = test_v - prev_v
spike  = 1 if abs(delta) > 10.0 else 0
noise  = test_v * 0.02

result = loaded_model.predict(
    loaded_scaler.transform([[test_v, v_sq, area, vc, ds, delta, spike, noise]])
)[0]
print(f"\nSpike test — {prev_v}V → {test_v}V (Δ={delta:+.1f}V ⚠️ spike)")
print(f"Result: {['🟢 SAFE','🟡 CHECK','🔴 DANGEROUS'][result]}")


In [ ]:
# CELL 18 — Export to Arduino C++ (ligtas_model.h)
import joblib, numpy as np
from micromlgen import port

rf_model = joblib.load("ligtas_rf_model.pkl")
scaler   = joblib.load("ligtas_scaler.pkl")

with open("ligtas_model.h", "w") as f:
    f.write(port(rf_model))
print("✅ ligtas_model.h created!")

means  = ", ".join([f"{m:.6f}f" for m in scaler.mean_])
scales = ", ".join([f"{s:.6f}f" for s in scaler.scale_])
print("\n=== COPY THESE INTO ligtas_ml.h ===")
print(f"const float SCALER_MEAN[8]  = {{{means}}};")
print(f"const float SCALER_SCALE[8] = {{{scales}}};")
print("\n// Feature order for input array:")
print("// [0] voltage_v          — raw sensor reading")
print("// [1] voltage_squared    — V²")
print("// [2] coverage_area_m2  — π × V²")
print("// [3] voltage_class      — 0-4")
print("// [4] danger_score       — combined risk")
print("// [5] voltage_delta      — change from previous reading  (GAP 4)")
print("// [6] spike_flag         — 1 if jump > 10V              (GAP 4)")
print("// [7] sensor_noise_level — 2% of voltage reading        (GAP 6)")
